In [ ]:
from _utils import *

# Load JSON data
json_df = pd.read_json("../data/raw/mod_files.json")

# Base output directory
base_dir = "../data/raw/nmodl_all"
os.makedirs(base_dir, exist_ok=True)

# Log directory and file
log_dir = "../download_logs"
log_file = os.path.join(log_dir, "failed_downloads.log")
os.makedirs(log_dir, exist_ok=True)

# Loop through the DataFrame rows
for _, row in tqdm(json_df.iterrows(), total=len(json_df), desc="Downloading ALL MOD files"):
    try:
        url = row["download_url"]
        file_hash = row["file_hash"]

        # Download and save as filehash.mod
        response = requests.get(url)
        response.raise_for_status()

        output_path = os.path.join(base_dir, f"{file_hash}.mod")
        with open(output_path, "wb") as f:
            f.write(response.content)

    except Exception as e:
        print(f"Error downloading {url}: {e}")
        with open(log_file, "a") as log:
            log.write(f"{file_hash},{url},{e}\n")

In [ ]:
# Define the folder path
mod5k_dir = Path("../data/raw/mod-5k")

# Get list of all files (adjust pattern if needed)
file_list = [f.name for f in mod5k_dir.glob("*") if f.is_file()]

# Create DataFrame
df_files = pd.DataFrame({"file_name": file_list})
df_files["file_hash"] = df_files["file_name"].str.replace(".mod", "", regex=False)


In [ ]:
df_files.shape

In [ ]:
link_df = df_files.merge(json_df, how="inner", on="file_hash")

In [ ]:
link_df2 = link_df[["file_hash","url"]]

In [ ]:
link_df2.to_csv("../data/mod_files_key.csv", index=False)